In [ ]:
import scipy.io
import numpy as np
import matplotlib.pyplot as plt
import mne
from scipy.signal import iirnotch, filtfilt, butter, welch
from mne.decoding import CSP
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import accuracy_score

DATA = r"C:\Users\DIVI\Downloads\data"
CHANNEL_NAMES = ['FC3','FCz','FC4','C5','C3','C1','Cz','C2','C4','C6','CP3','CP1','CPz','CP2','CP4','Pz']

def bandpass_filter(data, fs, lowcut=1, highcut=40, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [lowcut/nyq, highcut/nyq], btype='band')
    return filtfilt(b, a, data, axis=0)

def notch_filter(data, fs, freq, Q=30):
    b, a = iirnotch(freq, Q, fs)
    return filtfilt(b, a, data, axis=0)

def get_trigger_onsets(trig, fs):
    t = trig.flatten()
    onsets = np.where((t[1:] != 0) & (t[:-1] == 0))[0] + 1
    return onsets / fs

def epoch_data(data, trig, fs, time_after=2.0, period=6.0):
    sa, sp = int(time_after * fs), int(period * fs)
    trig_flat = trig.flatten()
    right, left = [], []
    for ot in get_trigger_onsets(trig, fs):
        idx = int(round(ot * fs))
        label = int(np.sign(np.sum(trig_flat[max(0, idx-2):idx+3])))
        s, e = idx + sa, idx + sa + sp
        if e <= data.shape[0]:
            ep = data[s:e, :]
            if label == -1: right.append(ep)
            elif label == 1: left.append(ep)
    return np.array(right), np.array(left)

def preprocess(mat, fs):
    y   = mat['y'].astype(np.float64)
    trg = mat['trig'].flatten()
    trim = int((get_trigger_onsets(trg, fs)[0] - 1) * fs)
    y, trg = y[trim:], trg[trim:]
    y = notch_filter(y, fs, 50)
    y = notch_filter(y, fs, 60)
    y = bandpass_filter(y, fs)
    return y, trg

def band_power(epochs, fs, lo, hi):
    out = []
    for ep in epochs:
        cp = []
        for ch in range(ep.shape[1]):
            f, Pxx = welch(ep[:, ch], fs=fs, nperseg=min(fs*2, ep.shape[0]))
            idx = (f >= lo) & (f <= hi)
            cp.append(np.trapezoid(Pxx[idx], f[idx]))
        out.append(cp)
    return np.mean(out, axis=0)

def erd_ers(active, ref, fs, lo, hi):
    return 10 * np.log10(band_power(active, fs, lo, hi) / band_power(ref, fs, lo, hi))

print("Setup done")


In [ ]:
keys = ['pre_training', 'pre_test', 'post_training', 'post_test']
mats = {k: scipy.io.loadmat(f"{DATA}\\P1_{k}.mat") for k in keys}
fs   = int(mats['pre_training']['fs'][0][0])

epochs = {}
for k, mat in mats.items():
    y, trg = preprocess(mat, fs)
    r, l   = epoch_data(y, trg, fs)
    epochs[k] = dict(right=r, left=l, y=y, trig=trg)
    print(f"{k:20s}  right={r.shape}  left={l.shape}")


In [ ]:
info = mne.create_info(ch_names=CHANNEL_NAMES, sfreq=fs, ch_types='eeg')
info.set_montage(mne.channels.make_standard_montage('standard_1020'))


In [ ]:
for band, lo, hi in [('Mu (8-12 Hz)', 8, 12), ('Beta (13-30 Hz)', 13, 30)]:
    fig, axes = plt.subplots(4, 2, figsize=(10, 14))
    for i, k in enumerate(['pre_training','pre_test','post_training','post_test']):
        lp = band_power(epochs[k]['left'],  fs, lo, hi)
        rp = band_power(epochs[k]['right'], fs, lo, hi)
        im, _ = mne.viz.plot_topomap(lp, info, axes=axes[i,0], show=False, cmap='Spectral_r')
        axes[i,0].set_title(f"{k} - LEFT",  fontsize=9)
        mne.viz.plot_topomap(rp, info, axes=axes[i,1], show=False, cmap='Spectral_r')
        axes[i,1].set_title(f"{k} - RIGHT", fontsize=9)
    fig.subplots_adjust(right=0.88)
    fig.colorbar(im, cax=fig.add_axes([0.9,0.15,0.02,0.7]), label='Power')
    fig.suptitle(f"{band} – P1", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
for band, lo, hi in [('Mu ERD/ERS (8-12 Hz)', 8, 12), ('Beta ERD/ERS (13-30 Hz)', 13, 30)]:
    fig, axes = plt.subplots(4, 2, figsize=(10, 14))
    erds, all_vals = [], []
    for k in ['pre_training','pre_test','post_training','post_test']:
        ref_r, ref_l = epoch_data(epochs[k]['y'], epochs[k]['trig'], fs, time_after=-1.0, period=1.0)
        el = erd_ers(epochs[k]['left'],  ref_l, fs, lo, hi)
        er = erd_ers(epochs[k]['right'], ref_r, fs, lo, hi)
        erds.append((k, el, er))
        all_vals += [el, er]
    vmax = max(abs(v).max() for v in all_vals)
    for i, (k, el, er) in enumerate(erds):
        im, _ = mne.viz.plot_topomap(el, info, axes=axes[i,0], show=False, cmap='RdBu_r', vlim=(-vmax, vmax))
        axes[i,0].set_title(f"{k} - LEFT",  fontsize=9)
        mne.viz.plot_topomap(er, info, axes=axes[i,1], show=False, cmap='RdBu_r', vlim=(-vmax, vmax))
        axes[i,1].set_title(f"{k} - RIGHT", fontsize=9)
    fig.subplots_adjust(right=0.88)
    fig.colorbar(im, cax=fig.add_axes([0.9,0.15,0.02,0.7]), label='ERD/ERS (dB)')
    fig.suptitle(f"{band} – P1", fontsize=14)
    plt.tight_layout()
    plt.show()


In [ ]:
import json as _json

def make_XY(right, left):
    X = np.concatenate([right, left], axis=0).transpose(0, 2, 1)
    y = np.array([0]*len(right) + [1]*len(left))
    return X, y

csp_results = {}
print("=== CSP + LDA — P1 ===")
for train_k, test_k, label in [('pre_training','pre_test','PRE '),
                                ('post_training','post_test','POST')]:
    X_tr, y_tr = make_XY(epochs[train_k]['right'], epochs[train_k]['left'])
    X_te, y_te = make_XY(epochs[test_k]['right'],  epochs[test_k]['left'])
    pipe = Pipeline([
        ('CSP', CSP(n_components=4, reg='ledoit_wolf', log=True, norm_trace=False)),
        ('LDA', LinearDiscriminantAnalysis())
    ])
    cv_acc = cross_val_score(pipe, X_tr, y_tr,
                             cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
                             scoring='accuracy')
    pipe.fit(X_tr, y_tr)
    y_pred = pipe.predict(X_te)
    test_acc = accuracy_score(y_te, y_pred)
    key = label.strip().lower()
    csp_results[key] = dict(cv=round(float(cv_acc.mean()),3),
                            cv_std=round(float(cv_acc.std()),3),
                            test=round(float(test_acc),3),
                            true=y_te.tolist(), pred=y_pred.tolist())
    print(f"{label} | CV: {cv_acc.mean():.3f} ± {cv_acc.std():.3f} | Test: {test_acc:.3f}")

out = rf"C:\\Users\\DIVI\\Downloads\\data\\P1_csp_results.json"
with open(out, 'w') as f:
    _json.dump(csp_results, f)
print(f"Results saved → {out}")
